In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re 
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

log_path = "data_logs/Aws Safi"
data_path = "data"

In [2]:
def add_three_hours(timestamp):
    k = timestamp.split('T')
    k[1]= str(int(k[1][:2])+3)+k[1][2:]
    return 'T'.join(k)
def mp(c):
    """Convert log color markers to valid Plotly colors"""
    if c.strip()=="'blue'":
        return 'blue'
    if c.strip()=="'lightblue'":
        return 'red'
    return 'black'

logs = {}
for log_file in os.listdir(log_path):
    if not log_file.endswith('.txt'):
        continue
    identifier = log_file.split("_run")[1].split(".")[0]
    try:
        with open(f"{log_path}/{log_file}", "r") as f:
            a = f.readlines()
            if not a:
                print(f"WARNING: Empty log file: {log_file}")
                continue
            logs[identifier] = [
                (datetime.strptime(add_three_hours(line.split(" ")[0]), "[%Y-%m-%dT%H:%M:%S.%fZ]").timestamp(),
                 mp(line.split(" ")[-1]),
                 " ".join(line.split(" ")[2:]))
                for line in a
                if 'blue' in line or '+' in line
            ]
    except Exception as e:
        print(f"ERROR parsing log file {log_file}: {str(e)}")

print(f"Successfully parsed {len(logs)} log files")

def validate_csv(file_path):
    """Validate if a CSV file is readable and has data"""
    try:
        if not os.path.exists(file_path):
            return False, "File does not exist"

        if os.path.getsize(file_path) == 0:
            return False, "File is empty"

        with open(file_path, 'r') as f:
            first_lines = []
            for i, line in enumerate(f):
                if i >= 5:
                    break
                first_lines.append(line)

            if not first_lines:
                return False, "File has no content"

            if not ',' in first_lines[0]:
                return False, "File does not appear to be a valid CSV"
        try:
            df_sample = pd.read_csv(file_path, nrows=5)
            if df_sample.empty:
                return False, "Pandas could not read any rows"
            return True, "Valid"
        except pd.errors.EmptyDataError:
            return False, "No columns to parse from file"
        except pd.errors.ParserError:
            return False, "Parser error - file may be corrupted"
        except Exception as e:
            return False, f"Error reading with pandas: {str(e)}"

    except Exception as e:
        return False, f"Validation error: {str(e)}"

Successfully parsed 3 log files


In [3]:
def match_timestamps_to_df_index(df, time_periods):
    """
    Efficiently match start and end timestamps to the closest times in the dataframe index.
    
    Parameters:
    df (pandas.DataFrame): DataFrame with a datetime index
    time_periods (list): List of tuples containing (start_time, end_time)
    
    Returns:
    list: List of tuples with matched (start_time, end_time) that exist in df.index
    """
    index_array = df.index.to_numpy()
    matched_periods = []
    
    for start_time, end_time in time_periods:
        start_dt = np.datetime64(pd.to_datetime(start_time))
        end_dt = np.datetime64(pd.to_datetime(end_time))

        start_pos = np.searchsorted(index_array, start_dt)
        end_pos = np.searchsorted(index_array, end_dt)
        
        if start_pos >= len(index_array):
            start_pos = len(index_array) - 1
        if end_pos >= len(index_array):
            end_pos = len(index_array) - 1
            
        start_time_matched = index_array[start_pos]
        end_time_matched = index_array[end_pos]
        
        matched_periods.append((start_time_matched, end_time_matched))
    
    return matched_periods

In [62]:
from datetime import datetime

# Given timestamp
timestamp = [1742289650.421065]
t = pd.Series(timestamp)

print(pd.to_datetime(t, unit='s',))
# Convert the timestamp to a datetime object
dt_object = datetime.fromtimestamp(timestamp[0])

# Print the readable datetime
print(dt_object)


0   2025-03-18 09:20:50.421065092
dtype: datetime64[ns]
2025-03-18 12:20:50.421065


In [4]:
dsets = {}
n = None
for file in os.listdir(data_path):
    if file.endswith(".csv"):
        identifier = ''.join(char for char in file.split("_")[0] if char.isdigit())
        if identifier not in logs:
            continue
        temp_df = pd.read_csv(f"{data_path}/{file}")
        temp_df.set_index('Timestamp', inplace=True)
        # temp_df.index = pd.to_datetime(temp_df.index, unit='s', utc=True)
        eeg_channels = [col for col in temp_df.columns if col.startswith('EEG.') and
                       not any(x in col for x in ['Counter', 'Interpolated', 'RawCq', 'Battery', 'MarkerHardware'])]
        temp_df = temp_df[eeg_channels]
        overt = {"start":[], "end":[]}
        labels = []
        for i,log in enumerate(logs[identifier]):
            if log[1] == 'red':
                word = log[2].split("'")[1]
                if word in {'Up','Down','Left','Right'}:
                    overt['start'].append(log[0])
                    overt['end'].append(logs[identifier][i+1][0])  
                    labels.append(word)
            
        idx = temp_df.index.to_numpy()
        n = idx
        overt_start_idx = np.searchsorted(idx, overt['start'])
        overt_end_idx = np.searchsorted(idx, overt['end'])       
        overt_series = []

        for start, end in zip(overt_start_idx, overt_end_idx):
            overt_series.append(temp_df.iloc[start:end])
        dsets[identifier] = {
            "overt": overt_series,
            "word": labels
        } 

In [67]:
n

array([1.74229116e+09, 1.74229116e+09, 1.74229116e+09, ...,
       1.74229184e+09, 1.74229184e+09, 1.74229184e+09])

In [5]:
[len(i) for i in dsets['1742289648917']['overt']]

[194,
 195,
 197,
 194,
 196,
 197,
 197,
 198,
 194,
 194,
 197,
 194,
 195,
 195,
 196,
 196,
 196,
 196,
 196,
 196,
 196,
 195,
 197,
 195,
 196]

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [7]:
one_hot = torch.eye(len(set(dsets['1742289648917']['word'])))
one_hot_dict = {word: one_hot[i] for i, word in enumerate(set(dsets['1742289648917']['word']))}

In [8]:
for item in dsets:
    dsets[item]['word'] = [one_hot_dict[x] for x in dsets[item]['word']]

In [9]:
class eeg_speech_dataset(Dataset):
    def __init__(self, dsets, identifiers, scaler=None, train=True):
        self.pairs = []
        self.scaler = scaler
        self.train = train
        
        for id in identifiers:
            overt_segments = dsets[id]['overt']
            word_labels = dsets[id]['word']
            for ovt, wrd in zip(overt_segments, word_labels):
                self.pairs.append((ovt.values, wrd))
        if self.scaler is None and self.train:
            all_data = np.vstack([p[0] for p in self.pairs])  # Only EEG data
            self.scaler = StandardScaler().fit(all_data)
            
    
    def __getitem__(self, idx):
        overt, word = self.pairs[idx]
        overt = self.scaler.transform(overt)
        return (torch.FloatTensor(overt), 
                torch.FloatTensor(word))
    def __len__(self):
        return len(self.pairs)


In [10]:
all_identifiers = list(dsets.keys())
train_ids, val_ids = train_test_split(all_identifiers, test_size=0.2, random_state=42)

train_dataset = eeg_speech_dataset(dsets, train_ids)
val_dataset = eeg_speech_dataset(dsets, val_ids, scaler=train_dataset.scaler, train=False)

In [ ]:
dsets.keys()  

dict_keys(['1742289648917', '1742290353356', '1742291163364'])

In [29]:
len(dsets['1742291163364']['overt'])

25

In [11]:
def collate_fn(batch):
    features = [item[0] for item in batch]  
    labels = [item[1] for item in batch]   
    
    lengths = torch.tensor([len(seq) for seq in features], dtype=torch.long)
    
    features_padded = nn.utils.rnn.pad_sequence(features, batch_first=True)
    
    labels_stacked = torch.stack(labels)
    
    return features_padded, labels_stacked, lengths

In [12]:
class EEGClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=2, dropout=0.2):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_dim,hidden_dim, num_layers, batch_first=True
        )
        
        self.classifier = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x, lengths):
        packed_x = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )
        packed_output, (hidden, _) = self.lstm(packed_x)
        last_hidden = hidden[-1]  
        output = self.classifier(last_hidden)
        
        return output

In [13]:
input_dim = train_dataset[0][0].shape[1] 
hidden_dim = 32 
num_classes = 4 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EEGClassifier(input_dim, hidden_dim, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

In [14]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for features, labels, lengths in loader:
        features, labels = features.to(device), labels.to(device)
        outputs = model(features, lengths)
        target_class = torch.argmax(labels, dim=1)
        loss = criterion(outputs, target_class)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == target_class).sum().item()
        total += labels.size(0)
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    
    print(f"Training - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")
    
    return avg_loss, accuracy
        
        
def validate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for features, labels, lengths in loader:
            features, labels = features.to(device), labels.to(device)
            
            outputs = model(features, lengths)
            
            target_class = torch.argmax(labels, dim=1)
            loss = criterion(outputs, target_class)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == target_class).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    
    print(f"Validation - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")
    
    return avg_loss, accuracy


In [15]:
def train_model(model, train_loader, val_loader, num_epochs=30, early_stopping_patience=5):
    model.to(device)
    best_val_loss = float('inf')
    early_stopping_counter = 0
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(model, train_loader)
        val_loss, val_acc = validate(model, val_loader)
        
        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_classifier.pt')
            early_stopping_counter = 0
        else:
            early_stopping_counter += 1
            
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        
        if early_stopping_counter >= early_stopping_patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break
    
    model.load_state_dict(torch.load('best_classifier.pt'))
    return model

In [16]:
batch_size=1
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=collate_fn)

model = EEGClassifier(input_dim, hidden_dim, num_classes).to(device)
model = train_model(model, train_loader, val_loader)

Training - Loss: 1.4263, Accuracy: 0.2200
Validation - Loss: 1.4220, Accuracy: 0.2000
Epoch 1/30 - Train Loss: 1.4263, Train Acc: 0.2200, Val Loss: 1.4220, Val Acc: 0.2000
Training - Loss: 1.4263, Accuracy: 0.2200
Validation - Loss: 1.4220, Accuracy: 0.2000
Epoch 2/30 - Train Loss: 1.4263, Train Acc: 0.2200, Val Loss: 1.4220, Val Acc: 0.2000
Training - Loss: 1.4263, Accuracy: 0.2200
Validation - Loss: 1.4220, Accuracy: 0.2000
Epoch 3/30 - Train Loss: 1.4263, Train Acc: 0.2200, Val Loss: 1.4220, Val Acc: 0.2000
Training - Loss: 1.4263, Accuracy: 0.2200
Validation - Loss: 1.4220, Accuracy: 0.2000
Epoch 4/30 - Train Loss: 1.4263, Train Acc: 0.2200, Val Loss: 1.4220, Val Acc: 0.2000
Training - Loss: 1.4263, Accuracy: 0.2200
Validation - Loss: 1.4220, Accuracy: 0.2000
Epoch 5/30 - Train Loss: 1.4263, Train Acc: 0.2200, Val Loss: 1.4220, Val Acc: 0.2000
Training - Loss: 1.4263, Accuracy: 0.2200
Validation - Loss: 1.4220, Accuracy: 0.2000
Epoch 6/30 - Train Loss: 1.4263, Train Acc: 0.2200, Va

C:\Users\Administrator\AppData\Local\Temp\ipykernel_4244\297704697.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_classifier.pt'